# AttentionMask-VLM — Kaggle Training Notebook
**2×T4 GPU | MSCOCO 2017 | open_clip ViT-B/16**

Before running:
- Add the **COCO 2017** dataset from Kaggle Datasets (path should resolve to `/kaggle/input/coco-2017`)
- Set accelerator to **GPU T4 x2** in notebook settings
- Add your W&B API key as a Kaggle Secret named `WANDB_API_KEY` (or set `USE_WANDB = False` below)

In [ ]:
# Cell 1: Install dependencies
!pip install -q open_clip_torch transformers datasets accelerate einops wandb

In [ ]:
# Cell 2: Environment check
import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.version.cuda}')
print(f'GPUs     : {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}  : {props.name}  {props.total_memory // 1024**3} GB')

In [ ]:
# Cell 3: Clone repo
import os
REPO_URL = 'https://github.com/Suyashkb/attentionmask-vlm'
REPO_DIR = '/kaggle/working/attentionmask-vlm'
if not os.path.exists(REPO_DIR):
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    print('Repo already cloned.')
os.chdir(REPO_DIR)
import sys
sys.path.insert(0, REPO_DIR)
print(os.listdir('.'))

In [ ]:
# Cell 4: W&B login
USE_WANDB = True
import os
if USE_WANDB:
    import wandb
    try:
        from kaggle_secrets import UserSecretsClient
        secret = UserSecretsClient().get_secret('WANDB_API_KEY')
        wandb.login(key=secret)
        print('W&B login OK')
    except Exception as e:
        print(f'Secret not found: {e}. Using offline mode.')
        os.environ['WANDB_MODE'] = 'offline'
else:
    os.environ['WANDB_MODE'] = 'disabled'
    print('W&B disabled.')

In [ ]:
# Cell 5: Verify COCO dataset
import os
COCO_ROOT = '/kaggle/input/coco-2017'
for split in ['train2017', 'val2017', 'annotations']:
    path = os.path.join(COCO_ROOT, split)
    exists = os.path.exists(path)
    count  = len(os.listdir(path)) if exists else 0
    print(f'{split}: {"OK" if exists else "MISSING"}  ({count} items)')

In [ ]:
# Cell 6: Load config
from train import load_config
cfg = load_config('configs/base.yaml')
cfg.data.data_dir    = COCO_ROOT
cfg.paths.output_dir = '/kaggle/working/checkpoints'
print('Config loaded')
print(f'  dataset={cfg.data.dataset}, batch={cfg.data.batch_size}, epochs={cfg.training.epochs}')

In [ ]:
# Cell 7: Smoke-test model (single forward pass — catches shape errors cheaply)
import torch
from model.attentionmask_vlm import AttentionMaskVLM
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = AttentionMaskVLM(cfg).to(device)
model.count_trainable_params()
B = 4
dummy_images = torch.randn(B, 3, 224, 224, device=device)
dummy_tokens = torch.randint(0, 49408, (B, 77), device=device)
with torch.no_grad():
    out = model(dummy_images, dummy_tokens)
print('Forward pass OK!')
for k, v in out.items():
    if hasattr(v, 'shape'):
        print(f'  {k}: {tuple(v.shape)}')
    else:
        print(f'  {k}: {float(v):.4f}')
del model, dummy_images, dummy_tokens
torch.cuda.empty_cache()

In [ ]:
# Cell 8: Smoke-test dataloader
from data.datasets import build_dataloaders
cfg.data.batch_size  = 8
cfg.data.num_workers = 2
train_loader, val_loader = build_dataloaders(cfg)
print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')
imgs, toks, caps = next(iter(train_loader))
print(f'Image: {tuple(imgs.shape)}  Token: {tuple(toks.shape)}')
print(f'Caption: {caps[0]}')

In [ ]:
# Cell 9: Full training run
cfg.data.batch_size  = 256   # reduce to 128 if OOM
cfg.data.num_workers = 4
from train import train
train(cfg)

In [ ]:
# Cell 10: Evaluate saved checkpoint
import torch
from model.attentionmask_vlm import AttentionMaskVLM
from data.datasets import build_dataloaders
from eval import evaluate
device = torch.device('cuda')
model  = AttentionMaskVLM(cfg).to(device)
ckpt_path = '/kaggle/working/checkpoints/best_model.pt'
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt['state_dict'])
print(f"Loaded checkpoint from epoch {ckpt['epoch']}")
_, val_loader = build_dataloaders(cfg)
metrics = evaluate(model, val_loader, device)
print('\nRetrieval metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v:.2f}')

In [ ]:
# Cell 11: Attention heatmap visualisation
import open_clip
from PIL import Image
from visualise import visualise_attention_heatmap
_, _, preprocess = open_clip.create_model_and_transforms('ViT-B-16', pretrained='openai')
tokenizer = open_clip.get_tokenizer('ViT-B-16')
# Replace with any COCO val image
image_path = '/kaggle/input/coco-2017/val2017/000000000139.jpg'
caption    = 'a dog running through a grassy field'
image  = preprocess(Image.open(image_path)).to(device)
tokens = tokenizer([caption])[0].to(device)
visualise_attention_heatmap(model, image, tokens)